# Section 02 lab — Reading data and schemas

Run the section setup once. Work through each activity from Step 1 onward; every step explains why it is needed before the code runs.


## Section setup — run once

Create one local Spark session and reuse it throughout this section.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Read CSV with default string types

Start with the simplest reader and inspect what Spark returns.


### Step 1 — Read the header row

The header option uses the first line as column names.


In [ ]:
transactions = spark.read.option('header', True).csv('../data/transactions.csv')


### Step 2 — Inspect the schema

Without inference or an explicit schema, CSV values remain strings.


In [ ]:
transactions.printSchema()


### Step 3 — Preview and count

Confirm both content and source size.


In [ ]:
transactions.show(5, truncate=False)
assert transactions.count() == 58


## Exercise 2 — Compare inferSchema with the default reader

Observe how Spark chooses types from the values it sees.


### Step 1 — Read the same file twice

Only the second reader requests schema inference.


In [ ]:
default_df = spark.read.option('header', True).csv('../data/transactions.csv')
inferred_df = (spark.read.option('header', True).option('inferSchema', True).csv('../data/transactions.csv'))


### Step 2 — Compare the schemas

Dirty values can prevent a column from becoming the numeric type you expected.


In [ ]:
print('Default schema')
default_df.printSchema()
print('Inferred schema')
inferred_df.printSchema()


### Step 3 — Inspect the inferred data

Schema inference does not replace business validation.


In [ ]:
inferred_df.show(5, truncate=False)
assert inferred_df.count() == 58


## Exercise 3 — Read CSV with an explicit schema

Declare the landing contract instead of relying on inference.


### Step 1 — Define the expected fields

Dirty numeric values intentionally remain strings for later validation.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
transaction_schema = StructType([
    StructField('transaction_id', StringType()), StructField('transaction_date', StringType()),
    StructField('customer_id', StringType()), StructField('product_id', StringType()),
    StructField('quantity', StringType()), StructField('unit_price', StringType()),
    StructField('status', StringType())])


### Step 2 — Apply the schema while reading

The reader now follows the declared contract.


In [ ]:
transactions = spark.read.schema(transaction_schema).option('header', True).csv('../data/transactions.csv')


### Step 3 — Verify the result

Check that the columns and types match the declaration.


In [ ]:
transactions.printSchema()
transactions.show(5, truncate=False)
assert transactions.count() == 58


## Exercise 4 — Read JSON and TSV

Use the same logical product columns from two physical formats.


### Step 1 — Define the product schema

A shared schema makes the two readers directly comparable.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
product_schema = StructType([
    StructField('product_id', StringType()), StructField('product_name', StringType()),
    StructField('category', StringType()), StructField('list_price', DoubleType())])


### Step 2 — Read newline-delimited JSON

Each line in the source is one JSON record.


In [ ]:
products_json = spark.read.schema(product_schema).json('../data/products.json')
products_json.show(5, truncate=False)


### Step 3 — Read tab-separated values

TSV uses the CSV reader with a tab separator.


In [ ]:
products_tsv = (spark.read.schema(product_schema).option('header', True).option('sep', '\t').csv('../data/products.tsv'))
products_tsv.show(5, truncate=False)
assert products_json.count() == products_tsv.count() == 12


## Exercise 5 — Write and read Parquet

Finish the ingestion path by publishing typed columnar data.


### Step 1 — Create typed rows in Python

A short inventory dataset makes the output easy to verify.


In [ ]:
rows = [('P101', 'Keyboard', 40), ('P102', 'Monitor', 12), ('P103', 'Notebook', 100)]
inventory = spark.createDataFrame(rows, ['product_id', 'product_name', 'stock'])
inventory.printSchema()


### Step 2 — Write the Parquet dataset

Overwrite keeps repeated runs deterministic inside the course output folder.


In [ ]:
output_path = '../output/section02_inventory_parquet'
inventory.write.mode('overwrite').parquet(output_path)


### Step 3 — Read the destination back

Validate the artifact that was actually written.


In [ ]:
saved = spark.read.parquet(output_path)
saved.printSchema()
saved.show()
assert saved.count() == 3
